In [1]:
import torch 
from torch import nn
from torch.nn import functional as F

In [2]:
from typing import Any


class Inception(nn.Module):
    """
    Inception module
    Args:
        c1: output channels cho nhánh 1 (1x1 conv)
        c2: tuple (reduce, out) cho nhánh 2 (1x1 conv -> 3x3 conv)
        c3: tuple (reduce, out) cho nhánh 3 (1x1 conv -> 5x5 conv)
        c4: output channels cho nhánh 4 (3x3 max pooling -> 1x1 conv)

    """

    def __init__(self, c1, c2, c3, c4):
        super().__init__()
        # branch 1
        self.b1_1 = nn.LazyConv2d(c1, kernel_size=1)

        # branch 2
        self.b2_1 = nn.LazyConv2d(c2[0], kernel_size=1)
        self.b2_2 = nn.LazyConv2d(c2[1], kernel_size=3, padding=1)
        
        # branch 3
        self.b3_1 = nn.LazyConv2d(c3[0], kernel_size=1)
        self.b3_2 = nn.LazyConv2d(c3[1], kernel_size=5, padding=2)
        
        # branch 4
        self.b4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.b4_2 = nn.LazyConv2d(c4, kernel_size=1)
        
    def forward(self, x):
        # Output shape: 
        b1 = F.relu(self.b1_1(x))
        b2 = F.relu(self.b2_2(F.relu(self.b2_1(x))))
        b3 = F.relu(self.b3_2(F.relu(self.b3_1(x))))
        b4 = F.relu(self.b4_2(self.b4_1(x)))
        return torch.cat((b1, b2, b3, b4), dim=1)
    

[!Note]
- Cần `ReLu` ở mỗi bước của forward để:
    - phục hồi tính phi tuyến do Conv2d tạo ra (4 layer conv ~ 1 layer conv lớn ~ 1 layer toán tuyến tính -> cần phi tuyến)
    - Mỗi nhánh đều độc lập -> cần ReLu riêng 
- Dùng `F.relu` thay vì `nn.ReLU` để tiết kiệm bộ nhớ (không cần lưu trữ tham số) và tăng tốc độ tính toán (không cần tạo đối tượng ReLU riêng cho mỗi nhánh)

In [3]:
from typing import Any


class GoogLeNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            self._b1(),
            self._b2(),
            self._b3(),
            self._b4(),
            self._b5(),
            nn.LazyLinear(num_classes),
        )

    def _b1(self):
        # STEM part 1: conv 7x7 giảm resolution nhanh  
        return nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

    def _b2(self):
        # STEM part 2: conv 1x1 bottleneck giảm số kênh trước khi conv 3x3
        return nn.Sequential(
            nn.LazyConv2d(64, kernel_size=1),
            nn.ReLU(),
            nn.LazyConv2d(192, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        
        

    def forward(self, x):
        return self.net(x)
    
    def _b3(self):
        # Body group 1: 2 inception + maxpool
        return nn.Sequential(
            Inception(64, (96, 128), (16, 32), 32),
            Inception(128, (128, 192), (32, 96), 64),
            # đóng vai trò giảm kích thước -> stride = 2 
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

    def _b4(self):
        # Body group 2: 5 inception + maxpool
        # C: 512 -> 512 -> 512 -> 528 -> 832 -> 832
        return nn.Sequential(
            Inception(192, (96, 208), (16, 48), 64),
            Inception(160, (112, 224), (24, 64), 64),
            Inception(128, (128, 256), (24, 64), 64),
            Inception(112, (144, 288), (32, 64), 64),
            Inception(256, (160, 320), (32, 128), 128),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        
    def _b5(self):
        # Body group 3: 2 inception + global avg pool
        return nn.Sequential(
            Inception(256, (160, 320), (32, 128), 128), # 832 C
            Inception(384, (192, 384), (48, 128), 128), # 1024 C
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

In [4]:
model = GoogLeNet()
X = torch.randn(1, 1, 96, 96)

for i, block in enumerate(model.net):
    X = block(X)
    print(f"Block {i} ({block.__class__.__name__:12s}) -> {str(X.shape):30s}")

Block 0 (Sequential  ) -> torch.Size([1, 64, 24, 24])   
Block 1 (Sequential  ) -> torch.Size([1, 192, 12, 12])  
Block 2 (Sequential  ) -> torch.Size([1, 480, 6, 6])    
Block 3 (Sequential  ) -> torch.Size([1, 832, 3, 3])    
Block 4 (Sequential  ) -> torch.Size([1, 1024])         
Block 5 (Linear      ) -> torch.Size([1, 10])           


1. Spatial: 96 -> 24 -> 12 -> 6 -> 3 -> 1
2. Channel: 64 -> 192 -> 480 -> 832 -> 1024 -> 1024

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((96, 96)),   # 96x96 thay vì 224x224
    transforms.ToTensor(),
])

train_data = datasets.FashionMNIST('./.data', train=True,
                                    download=True, transform=transform)
test_data  = datasets.FashionMNIST('./.data', train=False,
                                    transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=128)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GoogLeNet().to(device)

# Initialize Lazy modules first (materialize weights/bias shapes)
with torch.no_grad():
    _ = model(torch.zeros(1, 1, 96, 96, device=device))

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
model.apply(init_weights)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        y_hat = model(X)
        loss = F.cross_entropy(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct += (y_hat.argmax(1) == y).sum().item()
        total += y.size(0)
    
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            test_correct += (model(X).argmax(1) == y).sum().item()
    
    print(f"Epoch {epoch+1:2d} | "
          f"Loss: {total_loss/total:.4f} | "
          f"Train: {correct/total*100:.1f}% | "
          f"Test: {test_correct/len(test_data)*100:.1f}%")


Epoch  1 | Loss: 2.3005 | Train: 19.1% | Test: 38.5%
Epoch  2 | Loss: 2.2911 | Train: 35.6% | Test: 40.8%
Epoch  3 | Loss: 1.9765 | Train: 38.9% | Test: 49.4%
Epoch  4 | Loss: 0.9348 | Train: 62.9% | Test: 69.2%
Epoch  5 | Loss: 0.7375 | Train: 71.8% | Test: 73.4%
Epoch  6 | Loss: 0.6208 | Train: 76.7% | Test: 76.9%
Epoch  7 | Loss: 0.5532 | Train: 79.0% | Test: 79.1%
Epoch  8 | Loss: 0.5027 | Train: 81.0% | Test: 81.4%
Epoch  9 | Loss: 0.4734 | Train: 82.2% | Test: 74.3%
Epoch 10 | Loss: 0.4324 | Train: 83.9% | Test: 81.6%
